In [0]:
from databricks.feature_engineering import FeatureEngineeringClient
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lead

fe = FeatureEngineeringClient()
CATALOG = "mlo"
noaa_table = f"{CATALOG}.weather_mlops.noaa_historical_daily"
bronze = spark.table(noaa_table)

# Feature table — AWND/TMAX/TMIN only (PRCP excluded → no leakage)
feat = bronze.select("station", "date", "AWND", "TMAX", "TMIN").na.drop(subset=["station", "date"])
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.features")
FT = f"{CATALOG}.features.weather_daily"
spark.sql(f"DROP TABLE IF EXISTS {FT}")
fe.create_table(name=FT, primary_keys=["date", "station"], timestamp_keys=["date"],
                df=feat, description="Daily NOAA features (AWND,TMAX,TMIN)")

# Labels (kept separate — this is NOT in the feature table)
w = Window.partitionBy("station").orderBy("date")     # per-station, chronological

labels = (spark.table(noaa_table)
          .select("station", "date", "PRCP")
          .withColumn("Bad_today", (col("PRCP") > 0.5).cast("int"))
          .withColumn("Bad", lead("Bad_today", 1).over(w))   # ← Bad = TOMORROW's weather
          .na.drop(subset=["Bad"])                           # drop each station's last day
          .select("station", "date", "Bad"))

labels.write.mode("overwrite").saveAsTable(f"{CATALOG}.features.weather_labels")

print("Baseline feature table + labels built.")